In [0]:
df=spark.table("predictivemaintenancecatalog.gold_data.gold_data_historical_stratified")
display(df)

In [0]:
# add column named failure_mode to the table 
from pyspark.sql import functions as F

modes = ["OSF", "PWF", "RNF", "TWF", "HDF"]

df = df.withColumn(
    "failure_mode",
    F.concat_ws(
        ", ",
        *[
            F.when(F.col(c) == 1, F.lit(c))
            for c in modes
        ]
    )
)

df = df.withColumn(
    "failure_mode",
    F.when(
        F.col("failure_mode") == "",
        "None"
    ).otherwise(F.col("failure_mode"))
)

display(df)

In [0]:
# overwrite the original table

# 1. Create a temporary view of your current dataframe
df.createOrReplaceTempView("temp_df")

# 2. Run SQL to completely replace the table and its schema
spark.sql("""
    CREATE OR REPLACE TABLE predictivemaintenancecatalog.gold_data.gold_data_historical_stratified
    AS SELECT * FROM temp_df
""")


df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("predictivemaintenancecatalog.gold_data.gold_data_historical_stratified")



## Failure events by failure model

In [0]:
from pyspark.sql import functions as F 
failure_counts = df.agg(
    F.sum("OSF").alias("OSF"),
    F.sum("PWF").alias("PWF"),
    F.sum("RNF").alias("RNF"),
    F.sum("TWF").alias("TWF"),
    F.sum("HDF").alias("HDF")
)

display(failure_counts)

# In PySpark / Databricks
from pyspark.sql import DataFrame
is_spark_df = isinstance(failure_counts, DataFrame)

print(is_spark_df)

In [0]:
# reformat the failure_counts from wide to narrow
from pyspark.sql import functions as F
total_machines=df.count()
print(total_machines)

failure_chart = failure_counts.selectExpr(
    """
    stack(
        5,
        'OSF', OSF,
        'PWF', PWF,
        'RNF', RNF,
        'TWF', TWF,
        'HDF', HDF
    ) as (failure_mode, failure_count)
    """
)

failure_chart = failure_chart.groupBy("failure_mode").agg(
    F.sum("failure_count").alias("Failures")
)

total_failures = failure_chart.agg(F.sum("Failures").alias("total")).collect()[0]["total"]

failure_chart = failure_chart.withColumn(
    "Machine_type_Failure_Rate_%",
    F.round(F.col("Failures")/F.lit(total_failures)*100, 2)
)



display(failure_chart)

### save to table


In [0]:
# 1. Create a temporary view of your current dataframe
failure_chart.createOrReplaceTempView("temp_failure_chart")

# 2. Run SQL to completely replace the table and its schema
spark.sql("""
    CREATE OR REPLACE TABLE predictivemaintenancecatalog.gold_data.gold_failure_mode_summary
    AS SELECT * FROM temp_failure_chart
""")


In [0]:
failure_chart.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("predictivemaintenancecatalog.gold_data.gold_failure_mode_summary")

## Create a bridge
This bridge will be used to join the original table and the failure summary table

In [0]:
from pyspark.sql import functions as F

modes = ["OSF", "PWF", "RNF", "TWF", "HDF"]

df_bridge = df.select(
    "UDI",
    F.explode(
        F.array(*[
            F.when(F.col(c) == 1, F.lit(c))
            for c in modes
        ])
    ).alias("failure_mode")
).filter(
    F.col("failure_mode").isNotNull()
)

display(df_bridge)

In [0]:
df_joined = df_bridge.join(
    failure_chart,
    on="failure_mode",
    how="left"
)

display(df_joined)

In [0]:
# 1. Create a temporary view of your current dataframe
df_joined.createOrReplaceTempView("temp_df_joined")

# 2. Run SQL to completely replace the table and its schema
spark.sql("""
    CREATE OR REPLACE TABLE predictivemaintenancecatalog.gold_data.joined
    AS SELECT * FROM temp_df_joined
""")

df_joined.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("predictivemaintenancecatalog.gold_data.joined")

## failure events
One record per UDI and active failure mode; contains UDI, Type, and failure_mode

In [0]:
from pyspark.sql import functions as F

modes = ["OSF", "PWF", "RNF", "TWF", "HDF"]

df_events = df.select(
    "UDI",
    "Type",
    F.explode(
        F.array(*[
            F.when(F.col(c) == 1, F.lit(c))
            for c in modes
        ])
    ).alias("failure_mode")
).filter(F.col("failure_mode").isNotNull())

display(df_events)

In [0]:
# 1. Create a temporary view of your current dataframe
df_events.createOrReplaceTempView("temp_df_events")

# 2. Run SQL to completely replace the table and its schema
spark.sql("""
    CREATE OR REPLACE TABLE predictivemaintenancecatalog.gold_data.events
    AS SELECT * FROM temp_df_events
""")

df_events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("predictivemaintenancecatalog.gold_data.events")

## Failure rate by product type

In [0]:
from pyspark.sql import functions as F

Machine_type_Failure_Rate = df.groupBy("Type").agg(
    F.count("*").alias("Total"),
    F.sum("Machine_failure").alias("Failures"),
    F.round(F.col("Failures")/F.col("Total")*100,2).alias("Machine_type_Failure_Rate_%")
)

display(Machine_type_Failure_Rate)

In [0]:
Machine_type_Failure_Rate.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("predictivemaintenancecatalog.gold_data.Machine_type_Failure_Rate_summary")